# Follow-up: the 7 warnings from `python -m data_pipelines.quality.run_quality`

0 errors, 391 checks, 7 warnings -- confirms the `contracts.py` changes didn't break anything. Two of the warnings (`complaints.affected_product_id`/`digital_events.product_id` `product_owner_mismatch`) are the exact numbers already investigated and explained. This notebook digs into the **other five**, two of which look alarming at face value:

- `customers.registration_branch_id` foreign_key_orphans = **149,995/150,000** (99.997%!)
- `service_agents.assigned_branch_id` foreign_key_orphans = **831/1,200** (69.25%)
- `transactions` transaction_before_product_opening = 827,610/4,425,008 (18.7%)
- `products.product_number` unique_field_violations = 6/400,000
- `service_agents.employee_code` unique_field_violations = 13/1,200

A clue already sitting in the Bronze profile before running anything: `registration_branch_id` shows **150,000 distinct values across 150,000 customers** (every single one different), and `assigned_branch_id` shows **833 distinct values across 1,200 agents** -- both far exceeds the real `branches` table's 350 distinct `branch_id` values. That's the signature of IDs generated independently in the same string format rather than actually drawn from the real branch list -- the same class of issue as the already-solved `product_owner_mismatch` finding, potentially even more extreme (not even landing on a real branch most of the time, vs. landing on a real-but-wrong-owner product). Sections 1-2 confirm or refute this directly.

Run top to bottom.

In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    return con.execute(sql, params or []).df()

## 1. `customers.registration_branch_id` -- is it drawn from the real 350 branches at all?

In [ ]:
print("Distinct branch_id in bronze.branches:")
display(q("SELECT count(DISTINCT branch_id) AS n FROM bronze.branches"))
print("Distinct registration_branch_id in bronze.customers (vs. 150,000 customers):")
display(q("SELECT count(*) AS total_customers, count(DISTINCT registration_branch_id) AS distinct_values FROM bronze.customers"))
print("How many of those distinct values are ACTUALLY real branch_ids?")
display(q("""
    SELECT count(DISTINCT c.registration_branch_id) AS distinct_used,
           count(DISTINCT CASE WHEN b.branch_id IS NOT NULL THEN c.registration_branch_id END) AS distinct_that_are_real_branches
    FROM bronze.customers c LEFT JOIN bronze.branches b ON b.branch_id = c.registration_branch_id
"""))
print()
print("Sample of registration_branch_id values that ARE real branches (the 5 matches):")
display(q("""
    SELECT c.customer_id, c.registration_branch_id
    FROM bronze.customers c JOIN bronze.branches b ON b.branch_id = c.registration_branch_id
    LIMIT 10
"""))
print()
print("Format comparison -- real branch_id vs. registration_branch_id samples:")
display(q("SELECT branch_id FROM bronze.branches ORDER BY branch_id LIMIT 5"))
display(q("SELECT registration_branch_id FROM bronze.customers ORDER BY registration_branch_id LIMIT 5"))

## 2. `service_agents.assigned_branch_id` -- same check

In [ ]:
print("Distinct assigned_branch_id in bronze.service_agents (vs. 1,200 agents, vs. 350 real branches):")
display(q("""
    SELECT count(*) AS total_agents,
           count(assigned_branch_id) AS populated,
           count(DISTINCT assigned_branch_id) AS distinct_values
    FROM bronze.service_agents
"""))
print("How many of those distinct values are ACTUALLY real branch_ids?")
display(q("""
    SELECT count(DISTINCT a.assigned_branch_id) AS distinct_used,
           count(DISTINCT CASE WHEN b.branch_id IS NOT NULL THEN a.assigned_branch_id END) AS distinct_that_are_real_branches
    FROM bronze.service_agents a LEFT JOIN bronze.branches b ON b.branch_id = a.assigned_branch_id
    WHERE a.assigned_branch_id IS NOT NULL
""")) 

## 3. `transactions` transaction_before_product_opening -- how far before, and is it uniform or concentrated?

In [ ]:
print("Distribution of (opening_date - transaction_date) in days, for the affected rows:")
display(q("""
    SELECT
        min(date_diff('day', TRY_CAST(t.transaction_date AS TIMESTAMP)::DATE, TRY_CAST(p.opening_date AS DATE))) AS min_days_early,
        max(date_diff('day', TRY_CAST(t.transaction_date AS TIMESTAMP)::DATE, TRY_CAST(p.opening_date AS DATE))) AS max_days_early,
        avg(date_diff('day', TRY_CAST(t.transaction_date AS TIMESTAMP)::DATE, TRY_CAST(p.opening_date AS DATE))) AS avg_days_early,
        median(date_diff('day', TRY_CAST(t.transaction_date AS TIMESTAMP)::DATE, TRY_CAST(p.opening_date AS DATE))) AS median_days_early
    FROM bronze.transactions t JOIN bronze.products p ON t.product_id = p.product_id
    WHERE TRY_CAST(t.transaction_date AS TIMESTAMP)::DATE < TRY_CAST(p.opening_date AS DATE)
"""))
print()
print("Same product_owner_mismatch check for transactions, for contrast (expect 0, already confirmed earlier):")
display(q("""
    SELECT count(*) AS total_linked,
           count(*) FILTER (WHERE t.customer_id <> p.customer_id) AS owner_mismatches
    FROM bronze.transactions t JOIN bronze.products p ON t.product_id = p.product_id
    WHERE t.customer_id IS NOT NULL
"""))

## 4. `products.product_number` and `service_agents.employee_code` -- the small unique-field violations

In [ ]:
print("product_number duplicates (6 excess rows expected):")
display(q("""
    SELECT product_number, count(*) AS n FROM bronze.products
    WHERE product_number IS NOT NULL GROUP BY 1 HAVING count(*) > 1 ORDER BY n DESC
"""))
print()
print("employee_code duplicates (13 excess rows expected):")
display(q("""
    SELECT employee_code, count(*) AS n FROM bronze.service_agents
    WHERE employee_code IS NOT NULL GROUP BY 1 HAVING count(*) > 1 ORDER BY n DESC
"""))

In [ ]:
con.close()
print("closed")